In [1]:
from pathlib import Path
import time

SCRIPTS = Path('..') / 'ejemplos_codigo'
print('[OK] Entorno B10 listo')

[OK] Entorno B10 listo


# IA para el Desarrollo de Software y Agentes de Programación

Este bloque esta orientado a transformar la forma en que el equipo de desarrollo  
de la empresa trabaja dia a dia.

No se trata de generar snippets de código.  
Se trata de integrar un agente autónomo en el flujo de trabajo profesional.

**Actividad de calentamiento - Cronometra la tarea:**  
Se define una tarea sencilla: renombrar un método en 5 archivos.  
Un participante lo hace manualmente en el IDE.  
Otro lo hace con Claude Code.  
Se cronometran ambos y se discuten las diferencias.

**El cambio fundamental:**
```
Chatbot:  desarrollador escribe codigo  <--  herramienta de consulta pasiva
Agente:   desarrollador define intención  -->  agente actua, verifica, itera
```

> **Antes de seguir:** ¿en qué parte de tu flujo de trabajo diario como desarrollador inviertes más tiempo en tareas mecánicas que no requieren juicio técnico real?

<details>
<summary>Orientación para el instructor (desplegar tras la reflexión)</summary>

**Una respuesta madura menciona al menos uno de estos elementos:**
- Tareas de boilerplate: crear la estructura de un nuevo endpoint, escribir DTOs, generar tests para un servicio ya implementado
- Tareas de búsqueda: localizar dónde se usa una función, trazar el flujo de una funcionalidad a través de múltiples archivos
- Tareas de traducción: convertir una especificación verbal en código, adaptar un patrón existente a un nuevo contexto similar
- La distinción entre tareas donde el desarrollador aplica juicio (diseño de arquitectura, decisiones de seguridad) y tareas donde simplemente ejecuta un patrón conocido

**Si nadie responde, preguntar:**
"¿Cuánto tiempo tardáis en crear un nuevo endpoint en la empresa desde cero, incluyendo el controlador, el servicio, los DTOs y los tests? ¿Qué parte de ese tiempo es pensar y qué parte es escribir?"

**Señal de comprensión:**
El alumno puede separar las tareas mecánicas de las que requieren criterio real. Si además puede estimar cuánto tiempo representan esas tareas mecánicas en su semana, ha cuantificado el potencial de impacto del bloque antes de ver el código.

</details>

## 10.1 Agente vs Asistente de Chat

Un asistente de chat convencional:
- Responde dentro de una ventana de conversación aislada
- Necesita que copies y pegues fragmentos
- Carece de contexto sobre tu proyecto completo
- No puede ejecutar nada por ti

Un agente de programación como Claude Code:
- Opera directamente en tu terminal y tu sistema de archivos
- Tiene acceso al arbol de directorios completo
- Puede leer y escribir archivos
- Ejecuta comandos de shell, tests, linters
- Lee los resultados y toma decisiones sobre como proceder

**La diferencia no es de grado, es de naturaleza: un agente actua, un chatbot aconseja.**

| Dimensión | Asistente de Chat | Agente (Claude Code) |
|---|---|---|
| Contexto | Fragmentos copiados a mano | Proyecto completo en disco |
| Acción | Sugiere código en texto | Edita archivos directamente |
| Verificación | El desarrollador compila manualmente | Ejecuta tests y compila |
| Flujo | Copiar-pegar-adaptar | Automático con supervisión |
| Iteración | Requiere re-prompt con cada error | Detecta errores y reintenta |
| Escala | Un archivo a la vez | Multiples archivos en una sesión |

In [2]:
# Simulacion de las herramientas que usa un agente de programacion
# En Claude Code estas son llamadas reales al sistema de archivos

import os

def simular_glob(patron: str, base: Path = SCRIPTS) -> list:
    """Busca archivos por patron de nombre (como hace Claude Code internamente)."""
    if not base.exists():
        return []
    return sorted(base.glob(patron))

def simular_grep(patron_texto: str, base: Path = SCRIPTS, extension: str = '*.py') -> list:
    """Busca texto en archivos (como hace Claude Code internamente)."""
    import re
    resultados = []
    if not base.exists():
        return resultados
    for archivo in base.glob(extension):
        try:
            contenido = archivo.read_text(encoding='utf-8', errors='replace')
            for i, linea in enumerate(contenido.splitlines(), 1):
                if re.search(patron_texto, linea, re.IGNORECASE):
                    resultados.append((archivo.name, i, linea.strip()[:80]))
        except Exception:
            pass
    return resultados


print('=== Herramienta Glob: buscar archivos Python en ejemplos_codigo ===')
archivos = simular_glob('*.py')
if archivos:
    for a in archivos:
        print(f'  {a.name}')
else:
    # Demostrar con el directorio actual
    archivos = sorted(Path('.').glob('*.ipynb'))
    print('  (ejecutando en Jupyter_notebooks/)')
    for a in archivos[:5]:
        print(f'  {a.name}')

print()
print('=== Herramienta Grep: buscar definiciones de funciones ===')
resultados = simular_grep(r'def \w+', SCRIPTS)
if resultados:
    for archivo, linea, texto in resultados[:8]:
        print(f'  {archivo}:{linea}  {texto}')
else:
    print('  (sin resultados en el directorio de scripts)')
    print('  Ejemplo de lo que mostraria un grep real:')
    ejemplos_grep = [
        ('InventoryService.cs', 42, 'public async Task<Order> CreateAsync(OrderDto dto)'),
        ('OrderController.cs',  15, 'public IActionResult Post([FromBody] OrderRequest req)'),
        ('PricingService.cs',   88, 'private decimal Calculate(Product p, int qty)'),
    ]
    for f, l, t in ejemplos_grep:
        print(f'  {f}:{l}  {t}')

=== Herramienta Glob: buscar archivos Python en ejemplos_codigo ===
  00_separabilidad_sql_vs_ml.py
  00b_resumir_rle_vs_semantica.py
  01_neurona_simple.py
  02_capa_densa.py
  03_activaciones_relu_softmax.py
  04_loss_crossentropy.py
  05_backpropagation.py
  06_optimizadores.py
  07_red_completa_entrenamiento.py
  08_ml_clasico_sklearn.py

=== Herramienta Grep: buscar definiciones de funciones ===
  00b_resumir_rle_vs_semantica.py:54  def rle_encode(data: str) -> list[tuple[str, int]]:
  00b_resumir_rle_vs_semantica.py:69  def rle_decode(encoded: list[tuple[str, int]]) -> str:
  00b_resumir_rle_vs_semantica.py:111  def resumen_truncado(texto: str, n_palabras: int = 12) -> str:
  00b_resumir_rle_vs_semantica.py:124  def resumen_frecuencia(texto: str, top_n: int = 8) -> str:
  00b_resumir_rle_vs_semantica.py:132  def resumen_primera_ultima(texto: str) -> str:
  02_capa_densa.py:29  def spiral_data(samples, classes):
  02_capa_densa.py:101  def __init__(self, n_inputs, n_neurons):
  02

## 10.3 Generación Contextualizada

La clave de la generación de código efectiva con un agente es el contexto.

**Instrucción vaga vs instrucción precisa:**

```
Vaga:   "Crea un servicio"

Precisa: "Crea un servicio que siga el patron de IOrderService,
          inyecte IWarehouseRepository, y valide que el almacen
          de destino tiene capacidad suficiente"
```

**Flujo del agente ante una instrucción precisa:**
```
1. Lee InventoryController.cs -> entiende el patron existente
2. Lee IInventoryService.cs -> conoce los metodos disponibles
3. Lee los DTOs existentes -> reutiliza o crea los necesarios
4. Genera el endpoint siguiendo las convenciones del proyecto
5. Añade validación usando el patron de error handling existente
```

**Cuanto mas específica la instrucción y mas contexto tiene el agente,  
mejor sera el código generado.**

In [3]:
# Plantilla de instruccion para generacion de codigo contextualizada
# Demuestra la diferencia entre instruccion vaga y precisa

def evaluar_instruccion_agente(instruccion: str) -> dict:
    """
    Evalua la calidad de una instruccion para un agente de programacion.
    Criterios: especificidad, referencias a codigo existente, criterios de aceptacion.
    """
    score = 0
    detalles = []

    # Especificidad: menciona archivos, clases o metodos concretos?
    patrones_especificos = [
        r'\.cs\b', r'\.py\b', r'Controller', r'Service', r'Repository',
        r'endpoint', r'metodo', r'interfaz', r'clase', r'InjEct',
    ]
    import re
    especifico = any(re.search(p, instruccion, re.IGNORECASE) for p in patrones_especificos)
    if especifico:
        score += 30
        detalles.append('[OK] Menciona artefactos concretos del proyecto')
    else:
        detalles.append('[X]  Sin referencia a clases, archivos o metodos concretos')

    # Patron de referencia: sigue el patron de X
    patron_ref = re.search(r'sigue|patron|misma estructura|como en|igual que', instruccion, re.IGNORECASE)
    if patron_ref:
        score += 30
        detalles.append('[OK] Referencia a patron existente para imitar')
    else:
        detalles.append('[!]  Sin referencia a convenciones existentes')

    # Criterios de aceptacion: valida que, incluye test, manejo de errores
    criterios = re.search(r'valida|test|manejo|error|verifica|incluye|devuelve|retorna', instruccion, re.IGNORECASE)
    if criterios:
        score += 25
        detalles.append('[OK] Incluye criterios de aceptacion verificables')
    else:
        detalles.append('[!]  Sin criterios de aceptacion explicitos')

    # Longitud (una instruccion muy corta suele ser vaga)
    if len(instruccion) > 100:
        score += 15
        detalles.append('[OK] Longitud adecuada para capturar el contexto')
    else:
        detalles.append('[!]  Instruccion corta -> probablemente ambigua')

    nivel = 'POBRE' if score < 30 else 'BASICO' if score < 60 else 'BUENO' if score < 85 else 'EXCELENTE'
    return {'score': score, 'nivel': nivel, 'detalles': detalles}


# Comparacion de instrucciones
instrucciones = [
    ('Instruccion vaga',
     'Crea un endpoint para transferir stock'),

    ('Instruccion con patron',
     'Crea un endpoint POST /api/warehouses/{id}/transfer. Sigue el patron del endpoint en InventoryController.cs. Usa IInventoryService.'),

    ('Instruccion completa',
     'Crea un endpoint POST /api/warehouses/{id}/transfer que mueva stock entre almacenes. '
     'Sigue el patron del endpoint existente en InventoryController.cs. '
     'Usa el servicio IInventoryService para la logica de negocio. '
     'Anade validacion de que ambos almacenes existen. '
     'Maneja el error si el almacen origen no tiene stock suficiente con HTTP 422.'),
]

for nombre, instruccion in instrucciones:
    resultado = evaluar_instruccion_agente(instruccion)
    print(f'\n=== {nombre} ===')
    print(f'  Score: {resultado["score"]}/100  [{resultado["nivel"]}]')
    for d in resultado['detalles']:
        print(f'  {d}')


=== Instruccion vaga ===
  Score: 30/100  [BASICO]
  [OK] Menciona artefactos concretos del proyecto
  [!]  Sin referencia a convenciones existentes
  [!]  Sin criterios de aceptacion explicitos
  [!]  Instruccion corta -> probablemente ambigua

=== Instruccion con patron ===
  Score: 75/100  [BUENO]
  [OK] Menciona artefactos concretos del proyecto
  [OK] Referencia a patron existente para imitar
  [!]  Sin criterios de aceptacion explicitos
  [OK] Longitud adecuada para capturar el contexto

=== Instruccion completa ===
  Score: 100/100  [EXCELENTE]
  [OK] Menciona artefactos concretos del proyecto
  [OK] Referencia a patron existente para imitar
  [OK] Incluye criterios de aceptacion verificables
  [OK] Longitud adecuada para capturar el contexto


## 10.5 Automatización de Tests

La generación de tests es una de las aplicaciones mas inmediatamente útiles  
de un agente de programación.

**Flujo iterativo agente: Código -> Tests -> Fix -> Tests:**
```
1. Escribe o modifica codigo
2. Genera o actualiza los tests correspondientes
3. Ejecuta los tests
4. Si algun test falla, analiza el error y corrige
5. Repite hasta que todos los tests pasan
```

Este ciclo es el equivalente a TDD asistido:  
el desarrollador define la intención,  
el agente implementa y verifica,  
el desarrollador revisa el resultado final.

**Instrucción ejemplo para generación de tests en la empresa:**
```
> "Genera tests unitarios para OrderService.CreateAsync().
>  Cubre: flujo feliz, stock insuficiente, cliente no encontrado,
>  producto descatalogado, y precio calculado a cero.
>  Usa xUnit y Moq, siguiendo el patron de tests/OrderServiceTests.cs."
```

In [4]:
# Simulacion del ciclo de tests con el agente
# Muestra la estructura de un dataset de test y el flujo de verificacion

# --- Servicio simple de Python que el 'agente' va a testear ---
class StockService:
    """Servicio de stock simplificado (equivalente a IInventoryService en .NET)."""

    def __init__(self, stock_inicial: dict):
        self.stock = dict(stock_inicial)

    def get_stock(self, product_id: str) -> int:
        if product_id not in self.stock:
            raise KeyError(f'Producto no encontrado: {product_id}')
        return self.stock[product_id]

    def transferir(self, product_id: str, cantidad: int,
                   almacen_origen: str, almacen_destino: str) -> dict:
        if product_id not in self.stock:
            raise KeyError(f'Producto no encontrado: {product_id}')
        if cantidad <= 0:
            raise ValueError('La cantidad debe ser positiva')
        if self.stock[product_id] < cantidad:
            raise ValueError(f'Stock insuficiente: disponible={self.stock[product_id]}, solicitado={cantidad}')
        self.stock[product_id] -= cantidad
        return {
            'producto': product_id,
            'cantidad': cantidad,
            'origen': almacen_origen,
            'destino': almacen_destino,
            'stock_restante': self.stock[product_id],
        }


# --- Tests generados por el agente (equivalente a xUnit en .NET) ---
def run_tests():
    resultados = []

    def test(nombre: str, fn):
        try:
            fn()
            resultados.append(('[OK]', nombre))
        except AssertionError as e:
            resultados.append(('[FAIL]', f'{nombre}: {e}'))
        except Exception as e:
            resultados.append(('[ERROR]', f'{nombre}: {type(e).__name__}: {e}'))

    # Test 1: flujo feliz
    def test_transferencia_exitosa():
        svc = StockService({'REF-001': 100})
        resultado = svc.transferir('REF-001', 30, 'ALM-A', 'ALM-B')
        assert resultado['stock_restante'] == 70, f'Stock restante esperado: 70, obtenido: {resultado["stock_restante"]}'
        assert resultado['cantidad'] == 30
    test('transferencia_exitosa', test_transferencia_exitosa)

    # Test 2: stock insuficiente
    def test_stock_insuficiente():
        svc = StockService({'REF-001': 10})
        try:
            svc.transferir('REF-001', 50, 'ALM-A', 'ALM-B')
            raise AssertionError('Debia lanzar ValueError')
        except ValueError as e:
            assert 'insuficiente' in str(e).lower()
    test('stock_insuficiente_lanza_excepcion', test_stock_insuficiente)

    # Test 3: producto no encontrado
    def test_producto_no_encontrado():
        svc = StockService({'REF-001': 100})
        try:
            svc.transferir('REF-INEXISTENTE', 10, 'ALM-A', 'ALM-B')
            raise AssertionError('Debia lanzar KeyError')
        except KeyError:
            pass
    test('producto_no_encontrado_lanza_excepcion', test_producto_no_encontrado)

    # Test 4: cantidad cero o negativa
    def test_cantidad_invalida():
        svc = StockService({'REF-001': 100})
        for cantidad_invalida in [0, -5]:
            try:
                svc.transferir('REF-001', cantidad_invalida, 'ALM-A', 'ALM-B')
                raise AssertionError(f'Debia lanzar ValueError con cantidad={cantidad_invalida}')
            except ValueError:
                pass
    test('cantidad_invalida_lanza_excepcion', test_cantidad_invalida)

    # Test 5: transferencia exacta (stock queda en cero)
    def test_transferencia_total():
        svc = StockService({'REF-001': 50})
        resultado = svc.transferir('REF-001', 50, 'ALM-A', 'ALM-B')
        assert resultado['stock_restante'] == 0
    test('transferencia_stock_completo', test_transferencia_total)

    return resultados


print('=== Tests generados por el agente para StockService ===')
resultados = run_tests()
pasados = sum(1 for estado, _ in resultados if estado == '[OK]')
print()
for estado, nombre in resultados:
    print(f'  {estado} {nombre}')
print()
print(f'Resultado: {pasados}/{len(resultados)} tests pasados')
if pasados == len(resultados):
    print('[OK] Suite completa. El agente verifico la correccion del servicio.')
else:
    print('[!]  Hay tests fallando. El agente analizaria el error y aplicaria un fix.')

=== Tests generados por el agente para StockService ===

  [OK] transferencia_exitosa
  [OK] stock_insuficiente_lanza_excepcion
  [OK] producto_no_encontrado_lanza_excepcion
  [OK] cantidad_invalida_lanza_excepcion
  [OK] transferencia_stock_completo

Resultado: 5/5 tests pasados
[OK] Suite completa. El agente verifico la correccion del servicio.


## 10.7 Buenas Prácticas para Trabajar con Agentes

1. **Se específico con el contexto**  
   Mal: 'arregla el bug'  
   Bien: 'el endpoint POST /api/orders devuelve 500 cuando quantity es null, el error esta en OrderService.cs línea 142'

2. **Pide verificación**  
   Despues de cualquier cambio, solicitar que ejecute los tests.  
   No asumir que el cambio es correcto porque el agente lo dice.

3. **Revisa los diffs**  
   Antes de aceptar cambios, leer el diff generado.  
   El agente puede malinterpretar una intención ambigua.  
   La revisión humana es la última línea de defensa.

4. **Divide las tareas grandes**  
   Mal: 'refactoriza todo el módulo de pagos'  
   Bien: -> 'extrae la validación de tarjetas a un servicio independiente'  
         -> 'actualiza las referencias'  
         -> 'ejecuta los tests'

5. **Usa el historial de sesión**  
   Claude Code mantiene contexto durante toda la sesión.  
   Puedes referirte a cambios anteriores, pedir que deshaga algo,  
   o construir incrementalmente sobre lo ya hecho.

**El cambio mental clave:**  
El desarrollador pasa de ser quien escribe cada línea  
a ser quien define la intención, revisa los cambios propuestos  
y valida los resultados.  
El agente se encarga de la mecanica.

In [5]:
# Simulacion de inspeccion de codigo: busqueda de problemas comunes
# Equivalente a lo que haria un agente con herramientas Grep + Read

# Fragmentos de codigo C# simulados para analisis
codigo_fragmentos = [
    {
        'archivo': 'InventoryController.cs',
        'linea': 45,
        'codigo': 'string sql = "SELECT * FROM Products WHERE Name = '" + input + "'";',
        'problema': 'SQL_INJECTION',
    },
    {
        'archivo': 'PaymentController.cs',
        'linea': 12,
        'codigo': '[HttpPost]\npublic IActionResult Process([FromBody] PaymentDto dto)',
        'problema': 'MISSING_AUTHORIZE',
    },
    {
        'archivo': 'OrderService.cs',
        'linea': 88,
        'codigo': '_logger.LogInformation($"Card: {dto.CardNumber}, CVV: {dto.Cvv}");',
        'problema': 'SENSITIVE_DATA_LOG',
    },
    {
        'archivo': 'ProductRepository.cs',
        'linea': 33,
        'codigo': 'var products = await _context.Products.ToListAsync();  // sin paginacion',
        'problema': 'NO_PAGINATION',
    },
    {
        'archivo': 'CustomerController.cs',
        'linea': 20,
        'codigo': '[Authorize]\n[HttpGet]\npublic async Task<IActionResult> GetAll()',
        'problema': None,  # Codigo correcto
    },
]

# Reglas de inspeccion (equivalente a las instrucciones del agente)
REGLAS_INSPECCION = {
    'SQL_INJECTION': {
        'descripcion': 'Consulta SQL con concatenacion de strings (riesgo SQL injection)',
        'severidad': 'CRITICA',
        'solucion': 'Usar parametros: WHERE Name = @name',
    },
    'MISSING_AUTHORIZE': {
        'descripcion': 'Endpoint HTTP sin atributo [Authorize]',
        'severidad': 'ALTA',
        'solucion': 'Anadir [Authorize] o [AllowAnonymous] segun corresponda',
    },
    'SENSITIVE_DATA_LOG': {
        'descripcion': 'Datos sensibles (tarjeta, CVV) en logs',
        'severidad': 'CRITICA',
        'solucion': 'Eliminar datos de pago del log, usar solo los ultimos 4 digitos',
    },
    'NO_PAGINATION': {
        'descripcion': 'Consulta sin paginacion (riesgo de carga masiva)',
        'severidad': 'MEDIA',
        'solucion': 'Anadir .Skip(offset).Take(pageSize) o usar IQueryable con paginacion',
    },
}

print('=== Inspeccion de seguridad del modulo de Pagos y Inventarios ===')
print()
problemas_encontrados = 0
for fragmento in codigo_fragmentos:
    if fragmento['problema'] is None:
        print(f'[OK]   {fragmento["archivo"]}:{fragmento["linea"]}')
        continue

    regla = REGLAS_INSPECCION[fragmento['problema']]
    problemas_encontrados += 1
    sev = regla['severidad']
    marcador = '[CRITICO]' if sev == 'CRITICA' else '[ALTO]   ' if sev == 'ALTA' else '[MEDIO]  '
    print(f'{marcador} {fragmento["archivo"]}:{fragmento["linea"]}')
    print(f'         Problema: {regla["descripcion"]}')
    print(f'         Solucion: {regla["solucion"]}')
    print()

print(f'Resumen: {problemas_encontrados} problemas detectados en {len(codigo_fragmentos)} fragmentos')
print('El agente aplicaria las correcciones y volveria a ejecutar los tests.')

=== Inspeccion de seguridad del modulo de Pagos y Inventarios ===

[CRITICO] InventoryController.cs:45
         Problema: Consulta SQL con concatenacion de strings (riesgo SQL injection)
         Solucion: Usar parametros: WHERE Name = @name

[ALTO]    PaymentController.cs:12
         Problema: Endpoint HTTP sin atributo [Authorize]
         Solucion: Anadir [Authorize] o [AllowAnonymous] segun corresponda

[CRITICO] OrderService.cs:88
         Problema: Datos sensibles (tarjeta, CVV) en logs
         Solucion: Eliminar datos de pago del log, usar solo los ultimos 4 digitos

[MEDIO]   ProductRepository.cs:33
         Problema: Consulta sin paginacion (riesgo de carga masiva)
         Solucion: Anadir .Skip(offset).Take(pageSize) o usar IQueryable con paginacion

[OK]   CustomerController.cs:20
Resumen: 4 problemas detectados en 5 fragmentos
El agente aplicaria las correcciones y volveria a ejecutar los tests.


---
## 7. Ejercicio de Decisión: ¿usarias IA aquí?

### Caso: agente que genera tests unitarios automáticamente

El equipo de desarrollo de la empresa quiere usar un agente de IA para que escriba
automáticamente los tests unitarios de los módulos del ERP de la empresa.
El agente lee el código existente y genera los tests sin intervención humana.
En las pruebas, los tests generados cubren el 78% del código.

---

**Pregunta 1 - La métrica de cobertura**
¿78% de cobertura significa que los tests son buenos? ¿Que podria estar mal
aunque la cobertura sea alta?

**Pregunta 2 - Los tests que siempre revisa un senior**
¿En que partes del ERP deberia un desarrollador senior revisar obligatoriamente
los tests generados antes de aceptarlos?

**Pregunta 3 - Las zonas de exclusión**
¿Hay partes del sistema donde NO usarias este agente? ¿Por que?

**Pregunta 4 - El proceso para que mejore con el tiempo**
¿Como estructurarias el proceso de revisión de tests para que el sistema aprenda
de los rechazos sin acumular deuda técnica?

---
*Escribe tus respuestas en la celda siguiente.*

### Mis respuestas

**Pregunta 1 - La métrica de cobertura:**

*(escribe aquí)*

**Pregunta 2 - Los tests que siempre revisa un senior:**

*(escribe aquí)*

**Pregunta 3 - Las zonas de exclusión:**

*(escribe aquí)*

**Pregunta 4 - El proceso para que mejore con el tiempo:**

*(escribe aquí)*

---

<!--
CRITERIOS DE Evaluación (para el instructor)

Pregunta 1 - La métrica de cobertura:
Respuesta correcta: 78% de cobertura significa que el 78% de las líneas se ejecutan durante los tests,
pero no que los tests verifiquen algo útil.
Lo que puede estar mal aunque la cobertura sea alta:
 - Tests que ejecutan el código sin hacer assertions
 - Tests que verifican solo el camino feliz, no los casos de error
 - Tests que no cubren lógica de negocio crítica aunque si ejecuten el código
 - Assertions triviales (que algo no es None) sin verificar el comportamiento real
Insuficiente: "no, 78% es buena cobertura".

Pregunta 2 - Los tests que requieren revisión senior:
Areas críticas:
 - Lógica de cálculo de precios y facturacion (errores tienen impacto económico directo)
 - Integraciones con sistemas externos (ERP, banco, AEAT)
 - Validaciones de seguridad y permisos
 - Migraciones de datos
 - Casos limite documentados en incidencias anteriores
Insuficiente: "todos los tests" (respuesta trivial que no jerarquiza riesgo).

Pregunta 3 - Las zonas de exclusión:
Zonas donde NO usar el agente o usar con máxima precaucion:
 - Código de autenticación y autorización (errores de seguridad críticos)
 - Tests de integración con sistemas externos reales (riesgo de datos de producción)
 - Módulos con lógica regulatoria (contabilidad, fiscal, laboral)
 - Código legacy sin documentacion donde el agente no puede inferir el comportamiento esperado
Incorrecto: "no hay zonas de exclusión, el agente es suficientemente bueno".

Pregunta 4 - El proceso de mejora continua:
Proceso valido:
  1. Registrar cada test rechazado con el motivo específico
  2. Clasificar los rechazos por tipo (assertion incorrecta, caso de error no cubierto, etc.)
  3. Crear ejemplos de tests correctos para los patrones mas frecuentes
  4. Usar esos ejemplos como few-shot en el prompt del agente
  5. Medir si la tasa de rechazo disminuye con cada iteración
Insuficiente: "revisar los tests y corregirlos" sin un mecanismo de aprendizaje sistemático.
-->


## Resumen del Bloque 10

**La transición clave:**
```
Antes:  desarrollador escribe codigo -> IDE -> compilador -> tests
Ahora:  desarrollador define intención -> agente -> verifica -> desarrollador revisa
```

**Casos de uso de mayor ROI para el equipo .NET de la empresa:**

| Caso | Tiempo manual (est.) | Con agente |
|---|---|---|
| Renombrar método en 5 archivos | 10-15 min | 1-2 min |
| Generar tests para un servicio | 2-4 horas | 15-30 min |
| Trazar flujo de una funcionalidad | 30-60 min | 5-10 min |
| Revisión de seguridad de un módulo | 1-2 dias | 30-60 min |
| Generar endpoint + DTO + servicio | 1-2 horas | 20-40 min |

**El principio de la supervisión:**  
El agente actua, pero el desarrollador valida.  
La intención la define el humano.  
La mecanica la ejecuta el agente.  
El juicio final es siempre humano.

**Siguiente bloque (B11):** Orquestación, Agentes Colaborativos y Protocolo MCP - 
como conectar multiples agentes con las APIs existentes de la empresa.